# Data Pipeline Overview

This notebook explains the full data pipeline in this repository from raw MP-20 CSV files to KLDM-ready batches.

It covers:
1. Raw CSV and CIF parsing
2. Saving processed tensor splits
3. Turning clean crystals into graph samples
4. The DNG task: training data and sampling priors
5. The CSP task: training data and sampling priors
6. How those batches connect to forward diffusion, score prediction, reverse updates, and final sampling

The notebook only uses code from this repository. The goal is to make the data flow visible at every step with prints, tensor dumps, and a few plots.


## 0. Current Code Structure

The data package is intentionally small and split by responsibility:

- `mp_20parser.py`
  Reads raw MP-20 CSV files, parses CIF strings, and saves processed `.pt` splits.
- `dataset.py`
  Defines the three data sources:
  - `StoredCrystalDataset` for real processed crystals
  - `FormulaDataset` for CSP priors
  - `SizePriorDataset` for DNG priors
- `transformations.py`
  Defines one configurable `CrystalTransform` that:
  - adds the graph
  - projects lattice parameters into the training space
  - encodes atom types as integers or fixed-width one-hot vectors
- `tasks.py`
  Defines two task entry points:
  - `DNGTask`
  - `CSPTask`

So the pipeline is:

`raw CSV -> parsed crystal tensors -> dataset -> transform -> batched KLDM input`


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch

from kldm.data import (
    CSPTask,
    DNGTask,
    FormulaDataset,
    SizePriorDataset,
    StoredCrystalDataset,
    preprocess_csv,
    process_cif,
)
from kldm.data.transformations import training_transform, sampling_transform


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data' / 'mp_20').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')


ROOT = find_repo_root(Path.cwd())
DATA_DIR = ROOT / 'data' / 'mp_20'
TRAIN_CSV = DATA_DIR / 'train.csv'
VAL_CSV = DATA_DIR / 'val.csv'
TEST_CSV = DATA_DIR / 'test.csv'
TRAIN_PT = DATA_DIR / 'train.pt'
VAL_PT = DATA_DIR / 'val.pt'
TEST_PT = DATA_DIR / 'test.pt'
TRAIN_STATS = DATA_DIR / 'train_length_stats.json'

torch.manual_seed(0)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
def describe_tensor(name, value, max_rows=6):
    print(f"{name:>16}: shape={tuple(value.shape)}, dtype={value.dtype}")
    if value.ndim == 1:
        print(value[: max_rows * 2])
    else:
        print(value[:max_rows])


def describe_sample(title, sample):
    print(title)
    print('-' * len(title))
    print(sample)
    for key in ['pos', 'h', 'lengths', 'angles', 'l', 'edge_node_index']:
        if hasattr(sample, key):
            describe_tensor(key, getattr(sample, key))
    print()


def describe_batch(title, batch):
    print(title)
    print('-' * len(title))
    print(batch)
    for key in ['pos', 'h', 'lengths', 'angles', 'l', 'batch', 'edge_node_index']:
        if hasattr(batch, key):
            describe_tensor(key, getattr(batch, key))
    print()


def plot_structure(sample, title):
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection='3d')
    color = sample.h if sample.h.ndim == 1 else sample.h.argmax(dim=-1)
    scatter = ax.scatter(sample.pos[:, 0], sample.pos[:, 1], sample.pos[:, 2], c=color, cmap='viridis', s=55)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    fig.colorbar(scatter, ax=ax, label='species id')
    plt.show()


def before_after(title, before, after):
    print(title)
    print('=' * len(title))
    describe_sample('Before', before)
    describe_sample('After', after)

## 1. Raw MP-20 CSV Input

The parser begins with the MP-20 CSV files. The only required column is `cif`.

In [ ]:
pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'csv_exists': [TRAIN_CSV.exists(), VAL_CSV.exists(), TEST_CSV.exists()],
    'pt_exists': [TRAIN_PT.exists(), VAL_PT.exists(), TEST_PT.exists()],
    'stats_exists': [
        TRAIN_STATS.exists(),
        (DATA_DIR / 'val_length_stats.json').exists(),
        (DATA_DIR / 'test_length_stats.json').exists(),
    ],
})

In [ ]:
train_df = pd.read_csv(TRAIN_CSV, nrows=3)
print('Columns:', list(train_df.columns))
train_df.head(1)

## 2. Parsing One CIF

`process_cif(...)` turns one CIF string into a clean crystal sample. At this point there is no graph yet, no KLDM lattice projection yet, and no sampling prior logic yet.

In [ ]:
first_cif = train_df.loc[0, 'cif']
parsed = process_cif(first_cif)
describe_sample('Parsed MP-20 sample', parsed)
plot_structure(parsed, 'Parsed MP-20 crystal')

## 3. Preprocessing The Full Split

`preprocess_csv(...)` loops over the CSV splits and saves:
- `train.pt`, `val.pt`, `test.pt`
- `*_length_stats.json`

Those stats are later used by the training transform to stabilize the lattice representation.

In [ ]:
summary = preprocess_csv(DATA_DIR, output_folder=DATA_DIR, splits=('train', 'val', 'test'))
summary

In [ ]:
raw_train = StoredCrystalDataset(TRAIN_PT)
raw_sample = raw_train[0]
describe_sample('Stored crystal sample before transform', raw_sample)
print('Length-stat buckets:', len(json.loads(TRAIN_STATS.read_text())))

In [ ]:
num_atoms = [int(sample.pos.shape[0]) for sample in raw_train.samples[:500]]
length_a = [float(sample.lengths[0, 0]) for sample in raw_train.samples[:500]]
angle_alpha = [float(sample.angles[0, 0]) for sample in raw_train.samples[:500]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(num_atoms, bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Atoms per structure')
axes[1].hist(length_a, bins=20, color='darkorange', edgecolor='black')
axes[1].set_title('Lattice length a')
axes[2].hist(angle_alpha, bins=20, color='seagreen', edgecolor='black')
axes[2].set_title('Lattice angle alpha')
plt.tight_layout()
plt.show()

## 4. Training Transform: Before And After

The training transform does three important things:
- attach a complete graph as `edge_node_index`
- map the lattice to KLDM space and store it in `l`
- optionally change the species encoding

Below, the same stored crystal sample is shown before and after the transform.

In [ ]:
fit_transform = training_transform(length_stats_path=TRAIN_STATS)
train_dataset = StoredCrystalDataset(TRAIN_PT, transform=fit_transform)
train_sample = train_dataset[0]
before_after('Stored crystal -> training-ready crystal', raw_sample, train_sample)

In [ ]:
print('Raw lengths:', raw_sample.lengths)
print('Transformed lengths:', train_sample.lengths)
print('Raw angles:', raw_sample.angles)
print('Transformed angles:', train_sample.angles)
print('Graph edges shape:', train_sample.edge_node_index.shape)
print('Final lattice tensor l:', train_sample.l)

## 5. DNG Training Path

DNG training starts from real processed crystals.

Step by step:
1. `DNGTask.fit_dataset(path)` loads a processed split through `StoredCrystalDataset`.
2. The training transform adds:
   - `edge_node_index`
   - transformed `lengths` and `angles`
   - concatenated graph-level lattice `l`
   - optional fixed-width one-hot `h`
3. `DNGTask.datamodule(...)` batches those transformed crystals.
4. The model can then treat the batch as a clean state for forward diffusion.

This is the standard supervised training path for the de-novo task.


In [ ]:
dng_task = DNGTask()
dng_fit_dataset = dng_task.fit_dataset(TRAIN_PT)
describe_sample('One DNG training sample', dng_fit_dataset[0])

dng_fit_dm = dng_task.datamodule(TRAIN_PT, VAL_PT, TEST_PT, batch_size=2)
dng_fit_batch = next(iter(dng_fit_dm.train_dataloader()))
describe_batch('One DNG training batch', dng_fit_batch)

## 6. DNG Sampling Path

DNG reverse sampling does not start from a stored target crystal.
It starts from a prior over graph sizes.

Step by step:
1. `DNGTask.size_prior_from_split(...)` estimates how many atoms a crystal is likely to contain.
2. `DNGTask.sample_loader(...)` builds a `SizePriorDataset` from that distribution.
3. Each prior sample contains:
   - random fractional coordinates
   - a default atomic species prior
4. The sampling transform adds:
   - a fully connected graph
   - a default lattice prior `l`
5. The reverse diffusion sampler is expected to refine this prior into a crystal.


In [ ]:
size_prior = dng_task.size_prior_from_split(TRAIN_PT)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(min(len(size_prior), 25)), size_prior[:25], color='slateblue')
ax.set_title('DNG node-count prior from the training split')
ax.set_xlabel('number of nodes')
ax.set_ylabel('probability')
plt.show()

In [ ]:
raw_dng_prior = SizePriorDataset(size_prior, n_samples=4)
transformed_dng_loader = dng_task.sample_loader(size_prior, n_samples=4, batch_size=2)
transformed_dng_batch = next(iter(transformed_dng_loader))
before_after('DNG prior sample -> DNG sampling sample', raw_dng_prior[0], transformed_dng_batch[0])

## 7. CSP Training Path

CSP training is kept close to the DNG training path.

That means training still uses real processed crystals:
1. `CSPTask.fit_dataset(path)` loads real crystals.
2. The same training transform creates graph edges and the KLDM lattice representation.
3. Batching produces the exact tensor contract the model expects.

Why do this?
Because CSP sampling is conditioned on formulas, but supervised training still benefits from learning on real crystal structures and real lattice targets.


In [ ]:
csp_task = CSPTask()
csp_fit_dataset = csp_task.fit_dataset(TRAIN_PT)
describe_sample('One CSP training sample', csp_fit_dataset[0])

csp_fit_dm = csp_task.datamodule(TRAIN_PT, VAL_PT, TEST_PT, batch_size=2)
csp_fit_batch = next(iter(csp_fit_dm.train_dataloader()))
describe_batch('One CSP training batch', csp_fit_batch)

## 8. CSP Sampling Path

CSP sampling starts from composition only.

Step by step:
1. `FormulaDataset` parses formulas such as `SiO2` into atomic numbers.
2. It creates placeholder fractional coordinates for each atom.
3. The sampling transform adds:
   - a fully connected graph
   - a lattice prior `l`
   - optional one-hot atom encoding
4. The reverse diffusion process is then responsible for turning that conditional prior into a crystal structure.

So CSP training and CSP sampling intentionally start from different states:
- training uses real stored crystals
- sampling uses formula-conditioned priors


In [ ]:
raw_csp_prior = FormulaDataset(['SiO2', 'LiFePO4'], repeats=2)
transformed_csp_loader = csp_task.sample_loader(['SiO2', 'LiFePO4'], repeats=2, batch_size=2)
transformed_csp_batch = next(iter(transformed_csp_loader))
before_after('CSP formula prior -> CSP sampling sample', raw_csp_prior[0], transformed_csp_batch[0])

In [ ]:
plot_structure(raw_csp_prior[0], 'Raw CSP formula prior')
plot_structure(transformed_csp_batch[0], 'Transformed CSP sampling sample')

## 9. The KLDM Batch Contract

The data layer does not perform diffusion itself.
Its job is to produce a batch with the exact tensors the model needs.

After batching with PyG, both CSP and DNG produce the same core contract:
- `pos`: node positions
- `h`: atom representation
- `l`: graph-level lattice tensor
- `batch`: graph index per node
- `edge_node_index`: edge list

That common contract is what lets the same model code handle both tasks.


In [ ]:
def kldm_view(name, batch):
    print(name)
    print('-' * len(name))
    for key in ['pos', 'h', 'l', 'batch', 'edge_node_index']:
        describe_tensor(key, getattr(batch, key))
    print()


kldm_view('DNG fit batch', dng_fit_batch)
kldm_view('DNG sample batch', transformed_dng_batch)
kldm_view('CSP fit batch', csp_fit_batch)
kldm_view('CSP sample batch', transformed_csp_batch)

## 10. Connecting The Data To Forward Diffusion, Score Training, Reverse Steps, And Sampling

This section ties the data pipeline to the model pipeline.

### 10.1 Clean state vs prior state

The data layer produces two kinds of inputs:

- **clean training states**
  - DNG: real transformed crystals from `DNGTask.fit_dataset(...)`
  - CSP: real transformed crystals from `CSPTask.fit_dataset(...)`
- **sampling priors**
  - DNG: size-based priors from `DNGTask.sample_loader(...)`
  - CSP: formula-based priors from `CSPTask.sample_loader(...)`

### 10.2 What the forward process needs

A KLDM-style forward diffusion step typically consumes:
- `batch.pos` for coordinate / momentum diffusion
- `batch.l` for lattice diffusion
- `batch.h` for atom-type diffusion if enabled
- `batch.batch` to know which nodes belong to which graph
- `batch.edge_node_index` for the GNN

So the data layer is already responsible for everything *before* noise is applied.

### 10.3 What the score network sees

At training time, the model would perturb the clean state and then call the score network with something like:

- noisy positions `pos_t`
- noisy velocities `v_t`
- noisy or clean species `h_t`
- noisy lattice `l_t`
- `node_index=batch.batch`
- `edge_node_index=batch.edge_node_index`

### 10.4 What reverse sampling starts from

At inference time, reverse diffusion starts from the prior loaders:
- DNG prior batch: random positions, default species prior, lattice prior
- CSP prior batch: formula species, random positions, lattice prior

The sampler then updates those latent states across time steps until it reaches a final sample.

### 10.5 Code ideas: forward diffusion interface

The code below is not the full KLDM implementation. It shows the expected interface between the dataloader and a model.


In [ ]:
# Example: how a model could consume a DNG batch from this data pipeline.
encoded_task = DNGTask(species_mode='one_hot')
clean_batch = next(iter(encoded_task.datamodule(TRAIN_PT, VAL_PT, TEST_PT, batch_size=2).train_dataloader()))

def forward_diffusion_contract(batch, diffusion_v, diffusion_l, diffusion_h=None):
    graph_t = torch.rand(batch.num_graphs, 1)
    node_t = graph_t[batch.batch]

    if diffusion_v is None:
        pos_t = batch.pos
        v_t = torch.zeros_like(batch.pos)
        target_v = None
    else:
        (v_t, pos_t), target_v = diffusion_v.training_targets(node_t, batch.pos, index=batch.batch)

    if diffusion_l is None:
        l_t = batch.l
        target_l = None
    else:
        l_t, target_l = diffusion_l.training_targets(graph_t, batch.l)

    if diffusion_h is None:
        h_t = batch.h
        target_h = None
    else:
        h_t, target_h = diffusion_h.training_targets(node_t, batch.h)

    latents = {
        'pos': pos_t,
        'v': v_t,
        'h': h_t,
        'l': l_t,
        'node_index': batch.batch,
        'edge_node_index': batch.edge_node_index,
        't': graph_t,
    }
    targets = {'v': target_v, 'l': target_l, 'h': target_h}
    return latents, targets

print('One-hot DNG training sample:')
describe_sample('DNG sample with fixed-width one-hot h', encoded_task.fit_dataset(TRAIN_PT)[0])

print('Model-facing batch contract:')
kldm_view('One-hot DNG batch', clean_batch)


### 10.6 Code ideas: score network and reverse sampler

Once forward diffusion has produced noisy states, the score model and sampler can be organized around the same batch contract.

A typical loop is:
1. build noisy latents from a clean batch or prior batch
2. predict denoising outputs with the GNN
3. compute training losses during fitting
4. apply reverse updates during sampling

The data layer stays unchanged across both training and sampling. Only the state values change.


In [ ]:
# Example: pseudo-interface for score prediction and reverse sampling.

def score_model_step(net, latents):
    return net(
        t=latents['t'],
        pos=latents['pos'],
        v=latents['v'],
        h=latents['h'],
        l=latents['l'],
        node_index=latents['node_index'],
        edge_node_index=latents['edge_node_index'],
    )


def reverse_sampling_sketch(prior_batch, diffusion_v, diffusion_l, diffusion_h, net, n_steps=100):
    pos = prior_batch.pos
    v = torch.zeros_like(pos)
    h = prior_batch.h
    l = prior_batch.l
    node_index = prior_batch.batch
    edge_node_index = prior_batch.edge_node_index

    ts = torch.linspace(1.0, 1e-3, n_steps + 1)
    for i in range(n_steps):
        t = torch.full((prior_batch.num_graphs, 1), ts[i])
        dt = ts[i + 1] - ts[i]
        preds = net(
            t=t,
            pos=pos,
            v=v,
            h=h,
            l=l,
            node_index=node_index,
            edge_node_index=edge_node_index,
        )
        if diffusion_v is not None:
            pos, v = diffusion_v.reverse_step_em(t=t, dt=dt, pos_t=pos, v_t=v, pred_v_t=preds['v'], node_index=node_index)
        if diffusion_l is not None:
            l = diffusion_l.reverse_step(t=t, dt=dt, x_t=l, pred=preds['l'])
        if diffusion_h is not None:
            h = diffusion_h.reverse_step(t=t[node_index], dt=dt, x_t=h, pred=preds['h'])

    return {'pos': pos, 'h': h, 'l': l}

print('Key idea: the sampler reuses the same batch layout as training.')
print('Only the source of the state changes: clean crystals for training, priors for generation.')
